In [4]:
# Markov / State Transition Modeling for in progress journeys.
# AI assisted in writing (ChatGPT)
#
# Outputs:
#   - Top transitions (prob + count)
#   - Stuck/friction states (self-loops, leaving prob, entropy, dwell time if available)
#   - Dead-end states (most common last-observed)
#   - Expected remaining steps until censor end
#   - One-step stop probability

import re
import numpy as np
import pandas as pd


EVENTS_PATH = r"C:\Users\samvi\Downloads\open_journeys1.csv" #Change
EVENT_DEFS_PATH = None
USE_STAGE = False 

JOURNEY_COL = "id" 
EVENT_COL = "event_name"
TS_COL = "event_timestamp"
STEP_COL = "journey_steps_until_end" 

TOPK = 20
MIN_COUNT = 50

END_STATE = "__censored_end__"



def normalize_state(x) -> str:
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s

def choose_column(df: pd.DataFrame, preferred: str, fallbacks: list[str]) -> str:
    if preferred in df.columns:
        return preferred
    for c in fallbacks:
        if c in df.columns:
            return c
    raise KeyError(f"Could not find '{preferred}' or any of {fallbacks}. Columns={list(df.columns)}")

def read_event_defs(path: str) -> pd.DataFrame:
    ed = pd.read_csv(path)
    ed.columns = [c.strip() for c in ed.columns]
    return ed

def build_event_to_stage_map(ed: pd.DataFrame) -> dict:
    cols = {c.lower(): c for c in ed.columns}
    event_col_candidates = ["event_name", "event", "name"]
    stage_col_candidates = ["stage", "stage_name", "funnel_stage", "category", "group"]

    event_col = next((cols[c] for c in event_col_candidates if c in cols), None)
    stage_col = next((cols[c] for c in stage_col_candidates if c in cols), None)

    if event_col is None or stage_col is None:
        return {}

    mapping = {}
    tmp = ed[[event_col, stage_col]].dropna()
    for _, row in tmp.iterrows():
        mapping[normalize_state(row[event_col])] = normalize_state(row[stage_col])
    return mapping

def to_transition_probs(counts: pd.DataFrame) -> pd.DataFrame:
    return counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

def outgoing_entropy(P: pd.DataFrame) -> pd.Series:
    ent = {}
    for s in P.index:
        p = P.loc[s].values
        p = p[p > 0]
        ent[s] = float(-(p * np.log(p)).sum()) if len(p) else 0.0
    return pd.Series(ent, name="outgoing_entropy")

def top_transitions(P: pd.DataFrame, counts: pd.DataFrame, k: int, min_count: int) -> pd.DataFrame:
    rows = []
    for s in P.index:
        for t, p in P.loc[s].items():
            c = int(counts.loc[s, t]) if (s in counts.index and t in counts.columns) else 0
            if c >= min_count and p > 0:
                rows.append((s, t, float(p), c))
    out = pd.DataFrame(rows, columns=["from_state", "to_state", "prob", "count"])
    return out.sort_values(["prob", "count"], ascending=[False, False]).head(k) if not out.empty else out


df = pd.read_csv(EVENTS_PATH)

JOURNEY_COL = choose_column(df, JOURNEY_COL, ["id"])
EVENT_COL = choose_column(df, EVENT_COL, ["event_name"])
TS_COL = choose_column(df, TS_COL, ["event_timestamp"])

df[TS_COL] = pd.to_datetime(df[TS_COL], errors="coerce", utc=True)

df["_event"] = df[EVENT_COL].map(normalize_state)

event_to_stage = {}
if EVENT_DEFS_PATH is not None:
    ed = read_event_defs(EVENT_DEFS_PATH)
    event_to_stage = build_event_to_stage_map(ed)
    if USE_STAGE and not event_to_stage:
        print("[WARN] USE_STAGE=True but could not infer event->stage mapping from EVENT_DEFS_PATH. Falling back to events.")
        USE_STAGE = False

if USE_STAGE and event_to_stage:
    df["_state"] = df["_event"].map(lambda e: event_to_stage.get(e, "unknown_stage"))
    STATE_KIND = "stage"
else:
    df["_state"] = df["_event"]
    STATE_KIND = "event"

df = df.dropna(subset=[JOURNEY_COL, TS_COL, "_state"]).copy()

if STEP_COL in df.columns:
    df["_step"] = pd.to_numeric(df[STEP_COL], errors="coerce")
    df = df.sort_values([JOURNEY_COL, TS_COL, "_step"], kind="mergesort")
else:
    df = df.sort_values([JOURNEY_COL, TS_COL], kind="mergesort")

print("Loaded events:", len(df))
print("Unique journeys:", df[JOURNEY_COL].nunique())
print(f"Unique states ({STATE_KIND}):", df["_state"].nunique())

df["_next_state"] = df.groupby(JOURNEY_COL)["_state"].shift(-1)

trans = df.dropna(subset=["_next_state"])[["_state", "_next_state"]]
counts_obs = pd.crosstab(trans["_state"], trans["_next_state"])

last = df.groupby(JOURNEY_COL).tail(1)[["_state"]].copy()
last["_next_state"] = END_STATE
counts_end = pd.crosstab(last["_state"], last["_next_state"])

all_from = counts_obs.index.union(counts_end.index).union(pd.Index([END_STATE]))
all_to = counts_obs.columns.union(counts_end.columns).union(pd.Index([END_STATE]))

counts = counts_obs.reindex(index=all_from, columns=all_to, fill_value=0)
counts = counts.add(counts_end.reindex(index=all_from, columns=all_to, fill_value=0), fill_value=0).astype(int)

counts.loc[END_STATE, END_STATE] = max(1, counts.loc[END_STATE, END_STATE])

P = to_transition_probs(counts)

print("\nTransition matrix built.")
print("States:", len(P))

print("\n")
print("TOP TRANSITIONS")
topT = top_transitions(P, counts, k=TOPK, min_count=MIN_COUNT)
if topT.empty:
    print(f"No transitions with count >= {MIN_COUNT}. Try lowering MIN_COUNT.")
else:
    display(topT)



rows = []
for s in P.index:
    p_self = float(P.loc[s, s]) if s in P.columns else 0.0
    self_count = int(counts.loc[s, s]) if (s in counts.index and s in counts.columns) else 0
    out_total = int(counts.loc[s].sum()) if s in counts.index else 0
    rows.append((s, p_self, 1.0 - p_self, self_count, out_total))

stuck = pd.DataFrame(rows, columns=["state", "p_self", "p_leave", "self_count", "outgoing_count"]).set_index("state")

stuck = stuck.join(outgoing_entropy(P), how="left")

df["_next_ts"] = df.groupby(JOURNEY_COL)[TS_COL].shift(-1)
df["_dwell"] = (df["_next_ts"] - df[TS_COL]).dt.total_seconds()
dwell = df.dropna(subset=["_dwell"]).copy()
dwell = dwell[dwell["_dwell"] >= 0]

if len(dwell):
    g = dwell.groupby("_state")["_dwell"]
    dwell_stats = pd.DataFrame({
        "mean_dwell_seconds": g.mean(),
        "median_dwell_seconds": g.median(),
        "n_dwell_samples": g.size()
    })
    stuck = stuck.join(dwell_stats, how="left")

if END_STATE in stuck.index:
    stuck = stuck.drop(index=END_STATE)

stuck = stuck.fillna(0.0)

stuck["friction_score"] = (
    2.0 * stuck["p_self"]
    + 0.5 * np.log1p(stuck["self_count"])
    + 0.5 * np.log1p(stuck.get("mean_dwell_seconds", 0.0))
    + 0.5 * (1.0 / (1.0 + stuck["outgoing_entropy"]))
)

print("\n==============================")
print("STUCK / FRICTION STATES (Top)")
print("==============================")
display(stuck.sort_values("friction_score", ascending=False).head(TOPK))



print("\n==============================")
print("DEAD-END STATES (most common last observed)")
print("==============================")
last_states = df.groupby(JOURNEY_COL).tail(1)["_state"]
dead_end = last_states.value_counts(normalize=True).head(TOPK)
display(dead_end.to_frame("terminal_share"))


def expected_steps_to_end(P: pd.DataFrame, end_state: str) -> pd.Series:
    all_states = P.index.union(P.columns)
    P2 = P.reindex(index=all_states, columns=all_states, fill_value=0.0)

    if end_state not in P2.index:
        P2.loc[end_state] = 0.0
    if end_state not in P2.columns:
        P2[end_state] = 0.0
    P2.loc[end_state, :] = 0.0
    P2.loc[end_state, end_state] = 1.0

    row_sums = P2.sum(axis=1)
    nz = row_sums > 0
    P2.loc[nz] = P2.loc[nz].div(row_sums[nz], axis=0)

    transient = [s for s in P2.index if s != end_state]
    if len(transient) == 0:
        return pd.Series(dtype=float)

    Q = P2.loc[transient, transient].values
    I = np.eye(Q.shape[0])

    try:
        N = np.linalg.inv(I - Q)
    except np.linalg.LinAlgError:
        N = np.linalg.pinv(I - Q)

    t = (N @ np.ones((Q.shape[0], 1))).flatten()
    return pd.Series(t, index=transient, name="E_steps_until_censor_end")


estep = expected_steps_to_end(P, END_STATE)

print("\n")
print("EXPECTED REMAINING STEPS UNTIL CENSOR END")
display(estep.sort_values(ascending=False).head(TOPK).to_frame())
display(estep.sort_values(ascending=True).head(TOPK).to_frame())



print("\n")
print("ONE-STEP STOP PROBABILITY P(next = censor_end)")
if END_STATE in P.columns:
    p_stop = P[END_STATE].drop(index=[END_STATE], errors="ignore").sort_values(ascending=False).head(TOPK)
    display(p_stop.to_frame("p_stop_next"))
else:
    print("No censor_end column in transition matrix (unexpected).")



SAVE_PREFIX = "markov_out_"
counts.to_csv(f"{SAVE_PREFIX}transition_counts.csv")
P.to_csv(f"{SAVE_PREFIX}transition_probs.csv")
stuck.reset_index().to_csv(f"{SAVE_PREFIX}state_friction_metrics.csv", index=False)
dead_end.to_csv(f"{SAVE_PREFIX}terminal_share_top.csv")
estep.to_csv(f"{SAVE_PREFIX}expected_steps_until_censor_end.csv")

print("Saved outputs with prefix:", SAVE_PREFIX)


Loaded events: 6254175
Unique journeys: 158325
Unique states (event): 25

Transition matrix built.
States: 26


TOP TRANSITIONS


,from_state,to_state,prob,count
72,application_web_view,application_web_view,0.844153,1231497
60,application_web_submit,application_web_view,0.688934,94395
108,browse_products,browse_products,0.675486,1628178
39,application_web_approved,browse_products,0.610699,93997
22,add_to_cart,browse_products,0.528712,266529
98,begin_checkout,view_cart,0.525742,140372
213,promotion_created,promotion_created,0.514727,186101
50,application_web_declined,application_web_view,0.496669,2311
27,application_phone_approved,__censored_end__,0.489805,1153
195,pre-application_(3rd_party_affiliates),campaign_click,0.487787,4593



STUCK / FRICTION STATES (Top)


,p_self,p_leave,self_count,outgoing_count,outgoing_entropy,mean_dwell_seconds,median_dwell_seconds,n_dwell_samples,friction_score
state,,,,,,,,,
browse_products,0.675486,0.324514,1628178,2410380,1.049445,9.771970e+04,74.0,2361029.0,14.491361
promotion_created,0.514727,0.485273,186101,361553,1.610232,6.285385e+05,1.0,335031.0,13.963610
application_web_view,0.844153,0.155847,1231497,1458855,0.644418,1.857679e+04,1.0,1450968.0,13.919097
campaignemail_clicked,0.417942,0.582058,24300,58142,1.724413,7.811470e+05,61421.0,51116.0,12.852807
view_cart,0.261583,0.738417,192895,737415,1.492658,8.669760e+04,49.0,722333.0,12.493804
catalog_(mail),0.065236,0.934764,1690,25906,1.821758,3.272664e+06,1560107.5,25172.0,11.524761
begin_checkout,0.100978,0.899022,26961,266998,1.562833,1.499752e+05,128.0,257787.0,11.457261
campaign_click,0.011022,0.988978,1033,93721,1.929340,5.930061e+05,64800.0,72118.0,10.309807
place_order_phone,0.049086,0.950914,94,1915,1.943526,1.601086e+06,74870.0,1251.0,9.688072



DEAD-END STATES (most common last observed)


...

,E_steps_until_censor_end
application_web_view,47.668877
application_web_submit,45.724157
add_to_cart,41.334740
browse_products,40.945447
application_web_declined,40.085463
view_cart,40.004766
begin_checkout,39.076176
application_web_approved,37.725501
place_order_web,37.436286
catalog_(mail),37.203258


,E_steps_until_censor_end
application_phone_approved,17.419577
application_phone_declined,19.533609
account_downpaymentcleared,24.375637
place_order_phone,25.265118
account_downpaymentreceived,28.186566
site_registration,29.034747
pre-application_(3rd_party_affiliates),31.725671
campaign_click,32.085392
campaignemail_clicked,32.957496
fingerhut_university,34.920339




ONE-STEP STOP PROBABILITY P(next = censor_end)


,p_stop_next
application_phone_approved,0.489805
account_downpaymentcleared,0.408173
place_order_phone,0.346736
site_registration,0.276630
application_phone_declined,0.276596
campaign_click,0.230503
account_downpaymentreceived,0.218009
fingerhut_university,0.142857
pre-application_(3rd_party_affiliates),0.137957
campaignemail_clicked,0.120842


Saved outputs with prefix: markov_out_
